In [7]:
import pandas as pd
import pyarrow.parquet as pq

In [8]:
# path = "/Users/shingo.omori/Github.com/Developer/marker_model_training_and_inference/data/heavy/Mvalues_holdout_hard.parquet"
path = "data/heavy/beta_qced_projectedPC1-50Adj_imputed_ALL.parquet"

In [9]:
schema = pq.read_schema(path)
print(len(schema.names)) 

190


In [10]:
# cgから始まらないカラムだけ確認
schema = pq.read_schema(path)

# cg で始まらないカラムだけ
meta_cols = [name for name in schema.names if not name.startswith("cg")]

print(f"meta columns: {len(meta_cols)}")
for name in meta_cols:
    field = schema.field(name)
    print(f"  {name}\t{field.type}")

meta columns: 190
  cpgSite	string
  210095290019_R07C01	double
  210095290019_R16C01	double
  210095290019_R13C02	double
  210095290019_R03C01	double
  210095290019_R09C03	double
  210095290019_R08C02	double
  210095290019_R01C03	double
  210095290019_R10C03	double
  210095290019_R02C02	double
  210095290019_R08C03	double
  210095290019_R10C02	double
  210095290019_R16C02	double
  210095290019_R10C01	double
  210095290019_R13C03	double
  210095290019_R11C02	double
  210095290019_R04C03	double
  210095290019_R12C03	double
  210095290019_R07C03	double
  210095290019_R02C01	double
  210095290019_R06C02	double
  210095290019_R15C02	double
  210095290019_R01C01	double
  210095290019_R14C02	double
  210095290019_R03C02	double
  210095290019_R03C03	double
  210095290019_R12C02	double
  210095290019_R13C01	double
  210095290019_R11C01	double
  210095290019_R16C03	double
  210095290019_R05C01	double
  210095290019_R01C02	double
  210095290019_R15C03	double
  210095290019_R06C03	double
  210095

In [11]:
n_rows = 5
n_cg = 5  # 確認用に見る CpG 列数

schema = pq.read_schema(path)
meta_cols = [c for c in schema.names if not c.startswith("cg")]
cg_cols = [c for c in schema.names if c.startswith("cg")][:n_cg]

df = pd.read_parquet(path, columns=meta_cols + cg_cols).head(n_rows)

# サンプルIDは行インデックス（__index_level_0__）側に入っている
# df = df.reset_index().rename(columns={"index": "sample_id"})
print(df)

      cpgSite  210095290019_R07C01  210095290019_R16C01  210095290019_R13C02  \
0  cg00000109                0.936                0.923                0.932   
1  cg00000658                0.828                0.832                0.831   
2  cg00000714                0.221                0.216                0.265   
3  cg00000721                0.903                0.914                0.937   
4  cg00000897                0.522                0.535                0.526   

   210095290019_R03C01  210095290019_R09C03  210095290019_R08C02  \
0                0.892                0.906                0.935   
1                0.815                0.820                0.809   
2                0.221                0.225                0.205   
3                0.922                0.936                0.938   
4                0.519                0.534                0.533   

   210095290019_R01C03  210095290019_R10C03  210095290019_R02C02  ...  \
0                0.914               

In [5]:
df = pd.read_parquet(path, columns=['sample_id', 'cg00000714'])

In [6]:
df

,sample_id,cg00000714
0,209788070064_R10C02,0.197
1,209810840108_R11C01,0.245
2,209788070040_R15C03,0.204
3,209810840108_R03C01,0.288
4,209788070058_R08C02,0.179
...,...,...
432,209810840123_R01C02,0.209
433,209810840118_R15C01,0.208
434,209810840122_R13C02,0.217
435,209810840122_R01C01,0.218


In [7]:
df2 = pd.read_csv("data/ALL_metadata.csv")

In [9]:
df2

,個人ID,性別,年齢,msa_filename
0,9PZZJX91RlFI514F,男,41,209788070049_R15C02
1,EkyVgUorwQzZ2Z08,男,34,209788070049_R02C02
2,lITQCn6hD2RyEjnC,男,43,209788070049_R04C02
3,J6GVaX7L2D9x3l84,男,57,209788070049_R12C02
4,r65ZVUQwnKKw65g9,男,50,209788070049_R14C02
...,...,...,...,...
2289,UB6F7KzEtHBh2VMw,女,33,210095290015_R02C03
2290,8pzfjy7EHxrQ79HO,女,40,210095290015_R08C03
2291,AmHDnBZDfmJVEWRF,女,30,210095290015_R11C03
2292,UkAhQEp00VTWqP5S,女,33,210095290015_R13C03


In [12]:
df3 = df.merge(df2, left_on="sample_id", right_on="msa_filename", how="left")
df3 = df3[['msa_filename', '性別', '年齢']]
df3 = df3.rename(columns={"msa_filename": "sample_id", "性別": "SEX", "年齢": "AGE"})
df3.to_csv("data/ALL_metadata_for_holdout_hard.csv", index=False)

In [13]:
df_oike = pd.read_csv("data/oike_omics_all_predictions_with_clinical.csv")
df_mapping = pd.read_csv("data/kenshin_ID_SEX_AGE_20260302.csv")

In [15]:
df_mapping

,個人ID,性別,年齢,msa_filename
0,9PZZJX91RlFI514F,男,41,209788070049_R15C02
1,EkyVgUorwQzZ2Z08,男,34,209788070049_R02C02
2,lITQCn6hD2RyEjnC,男,43,209788070049_R04C02
3,J6GVaX7L2D9x3l84,男,57,209788070049_R12C02
4,r65ZVUQwnKKw65g9,男,50,209788070049_R14C02
...,...,...,...,...
2289,UB6F7KzEtHBh2VMw,女,33,210095290015_R02C03
2290,8pzfjy7EHxrQ79HO,女,40,210095290015_R08C03
2291,AmHDnBZDfmJVEWRF,女,30,210095290015_R11C03
2292,UkAhQEp00VTWqP5S,女,33,210095290015_R13C03


In [14]:
df_oike

,SampleID,OID40001_ABCF3,OID40002_ABHD5,OID40003_ABHD8,OID40004_ACAD9,OID40005_ACSM5,OID40006_ADGRF2,OID40007_AFTPH,OID40008_AHI1,OID40009_AJM1,...,hematocrit,platelet_count,mcv,mch,mchc,fasting_blood_glucose,hba1c_ngsp,systolic_bp_avg,diastolic_bp_avg,cigarettes_pack_year
0,0ivdqv0qt7i0xBLI,0.212498,0.666031,1.848221,3.066847,3.457779,2.105532,1.108964,1.773811,0.376400,...,42.998450,3.167940,93.922170,30.918222,32.99414,4.567098,1.703289,4.792923,4.319161,5.272364
1,VGXEG6XluL30RLg9,0.212498,0.666031,1.926134,3.021511,3.457779,2.105532,1.108964,1.773811,0.379601,...,42.759480,3.201972,93.042620,30.443556,32.99414,4.555717,1.703801,4.793556,4.311649,5.221735
2,fPwLAZHmG4wkdOZp,0.212498,0.666031,2.089205,1.881112,3.457779,2.105532,1.108964,1.773811,0.377048,...,42.821490,3.166320,92.907310,30.624527,32.99414,4.561173,1.693767,4.785323,4.300934,5.495451
3,LzhTSoim7J8APgnh,0.212498,0.666031,2.356211,2.530974,3.457779,2.105532,1.108964,1.773811,0.379924,...,42.168870,3.182526,93.865135,30.549381,32.99414,4.548922,1.694995,4.749449,4.278512,5.319025
4,azOWTXprEANhLEY5,0.212498,0.666031,3.415209,2.594134,3.457779,2.105532,1.108964,1.773811,0.380275,...,41.783455,3.213068,93.870800,30.969728,32.99414,4.544902,1.680992,4.763079,4.289777,5.057590
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2208,4fDyYr5Q6gyMnStF,0.212498,0.666031,2.709920,2.195778,3.457779,2.105532,1.108964,1.773811,0.378915,...,42.602673,3.200407,93.332430,30.785530,32.99414,4.548069,1.684828,4.779090,4.304188,5.614089
2209,emMCJS6zdydAfDmu,0.212498,0.666031,2.311218,2.162135,3.457779,2.105532,1.108964,1.773811,0.380140,...,42.320510,3.328069,89.323090,29.658907,32.99414,4.538478,1.679961,4.755735,4.277972,4.865296
2210,0qjE07I3oybAf5zz,0.212498,0.666031,2.228514,2.577714,3.457779,2.105532,1.108964,1.773811,0.378843,...,41.713856,3.138801,93.780730,31.019882,32.99414,4.553373,1.697740,4.757858,4.283846,5.050490
2211,weendqVYL4OqJpxo,0.212498,0.666031,1.958633,2.168510,3.457779,2.105532,1.108964,1.773811,0.376401,...,42.809360,3.233803,92.646030,30.505476,32.99414,4.548489,1.678295,4.761799,4.294085,5.228726


In [21]:
df_oike_identified = df_oike.merge(df_mapping, left_on="SampleID", right_on="個人ID", how="inner")

In [22]:
df_oike_identified.to_csv("data/oike_dcEMA_identified.csv", index=False)

In [35]:
df_oike_identified = pd.read_csv("data/oike_dcEMA_identified.csv")

In [ ]:
grim_map = pd.read_csv("data/EMAmarker_Grimage_mapping.csv")
df_oike_identified = df_oike_identified[['msa_filename', '性別', '年齢'] + grim_map['EMAmarker'].tolist()]
df_oike_identified = df_oike_identified.rename(columns={"msa_filename": "sample_id", "性別": "SEX", "年齢": "AGE"})
df_oike_identified.to_csv("data/oike_dcEMA_on_grimagev2_markers.csv", index=False)

In [42]:
df_oike_identified

,sample_id,SEX,AGE,cigarettes_pack_year,OID43339_ADM,OID45441_B2M,OID45345_CST3,OID45131_GDF15,OID44746_LEP,crp,hba1c_ngsp,OID45256_SERPINE1,OID45425_TIMP1
0,209788070068_R09C03,男,31,5.272364,0.097565,0.035593,0.095715,0.290303,-0.126927,-2.581937,1.703289,0.240196,0.077696
1,210095290011_R13C03,男,46,5.221735,0.145514,0.067427,0.076336,0.245145,-0.103573,-2.783693,1.703801,0.245835,0.084740
2,209810840106_R07C01,女,54,5.495451,0.143684,0.081050,0.144972,0.434770,-0.179061,-2.634757,1.693767,0.067569,0.100604
3,209788070055_R09C01,女,55,5.319025,0.058907,0.066653,0.030577,0.309274,-0.418912,-2.833944,1.694995,-0.080326,0.038317
4,209810840021_R05C03,女,45,5.057590,0.027075,0.039989,0.032096,0.081179,0.037384,-2.747380,1.680992,-0.005269,0.022349
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2198,209810840123_R01C02,男,59,5.614089,0.048847,-0.017387,-0.012392,-0.018143,-0.026153,-2.586675,1.684828,0.260745,0.006764
2199,209810840118_R15C01,男,43,4.865296,-0.152802,0.001022,-0.116614,-0.311787,0.131147,-2.591207,1.679961,-0.137257,-0.036672
2200,209810840122_R13C02,女,51,5.050490,0.012028,0.011251,-0.009308,0.056966,0.224437,-2.691228,1.697740,0.126802,0.027963
2201,209810840122_R01C01,男,51,5.228726,-0.048514,-0.029493,0.019606,0.027526,-0.073110,-2.681230,1.678295,0.007561,0.029215


In [23]:
df_grim = pd.read_csv("output/grimage_v2_wide.csv")

In [24]:
df_grim.columns

Index(['sample_id', 'DNAmPACKYRS', 'DNAmADM', 'DNAmB2M', 'DNAmCystatinC',
       'DNAmGDF15', 'DNAmLeptin', 'DNAmlogCRP', 'DNAmlogA1C', 'DNAmPAI1',
       'DNAmTIMP1', 'GrimAgeV2', 'AgeAccelGrimV2'],
      dtype='str')

In [ ]:
df_obs = pd.read_excel("data/kenshin_observed_data.xlsx")
df_obs = df_obs[['個人ID','CRP定量','HbA1c(NGSP)','たばこについて','禁煙時期（年数）','喫煙本数（本／日）','喫煙期間（年数）']]

In [50]:
df_oid_obs = pd.read_csv("data/OID_joined.csv")

In [51]:
df_oid_obs

,sample_id,OID43339_ADM_observed,OID45441_B2M_observed,OID45345_CST3_observed,OID45131_GDF15_observed,OID44746_LEP_observed,OID45256_SERPINE1_observed,OID45425_TIMP1_observed
0,01BVFyvQ1okMLGqm,-0.379073,-0.087856,-0.015910,-0.240096,-2.542354,0.124804,-0.154058
1,0BZFK1m1aOUDbEQp,-0.052648,-0.020133,0.044221,-0.074595,-1.278558,-1.009167,-0.207357
2,0D0oB1FnJQ3d6C3M,-0.341382,0.110012,-0.033772,-0.409081,1.083360,-0.221919,0.104515
3,0DLrEIGZCTETKAtT,-0.063122,-0.233871,-0.584804,-0.789165,0.304766,0.644313,-0.343274
4,0E6NcFfrWPRZVFf9,-0.088406,-0.143026,-0.066229,0.193179,1.099255,-1.419803,-0.314817
...,...,...,...,...,...,...,...,...
2208,zrrx790EHTcw9PKr,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2209,zttw3VRwevmSDUOw,0.358983,0.804410,0.554036,1.108380,1.258896,1.673386,0.462905
2210,zvFLRCb2nJLuescM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2211,zvFgjPzGM3qkUoTA,-0.173059,-0.167677,-0.477717,-0.127751,-0.176879,0.629214,-0.230596


In [52]:
df_obs = df_obs.merge(df_oid_obs, left_on="個人ID", right_on="sample_id", how="inner")

In [60]:
df_obs_identified = df_obs.merge(df_mapping, left_on="個人ID", right_on="個人ID", how="inner")
df_obs_identified.drop(columns=["個人ID","sample_id"], inplace=True)
df_obs_identified.rename(columns={"msa_filename":"sample_id", "性別":"SEX", "年齢":"AGE"}, inplace=True)

In [61]:
df_obs_identified.columns

Index(['CRP定量', 'HbA1c(NGSP)', 'たばこについて', '禁煙時期（年数）', '喫煙本数（本／日）', '喫煙期間（年数）',
       'OID43339_ADM_observed', 'OID45441_B2M_observed',
       'OID45345_CST3_observed', 'OID45131_GDF15_observed',
       'OID44746_LEP_observed', 'OID45256_SERPINE1_observed',
       'OID45425_TIMP1_observed', 'SEX', 'AGE', 'sample_id'],
      dtype='str')

In [62]:
cols = ['sample_id', 'SEX', 'AGE','CRP定量', 'HbA1c(NGSP)', 'たばこについて', '禁煙時期（年数）', '喫煙本数（本／日）', '喫煙期間（年数）',
       'OID43339_ADM_observed', 'OID45441_B2M_observed',
       'OID45345_CST3_observed', 'OID45131_GDF15_observed',
       'OID44746_LEP_observed', 'OID45256_SERPINE1_observed',
       'OID45425_TIMP1_observed']
df_obs_identified = df_obs_identified[cols]

In [ ]:
df_obs_identified['pack_years'] = df_obs_identified['喫煙本数（本／日）'] / 20 * df_obs_identified['喫煙期間（年数）']
df_obs_identified['pack_years'] = df_obs_identified['pack_years'].fillna(0)

In [69]:
df_obs_identified.drop(columns=['たばこについて', '禁煙時期（年数）', '喫煙本数（本／日）', '喫煙期間（年数）'], inplace=True)

In [70]:
df_obs_identified

,sample_id,SEX,AGE,CRP定量,HbA1c(NGSP),OID43339_ADM_observed,OID45441_B2M_observed,OID45345_CST3_observed,OID45131_GDF15_observed,OID44746_LEP_observed,OID45256_SERPINE1_observed,OID45425_TIMP1_observed,pack_years
0,210095290011_R03C03,男,52,0.05以下,5.6,-0.379073,-0.087856,-0.015910,-0.240096,-2.542354,0.124804,-0.154058,0.0
1,209798720002_R05C02,女,39,NaN,5.2,-0.052648,-0.020133,0.044221,-0.074595,-1.278558,-1.009167,-0.207357,0.0
2,209810840119_R07C02,女,38,0.05以下,5.5,-0.341382,0.110012,-0.033772,-0.409081,1.083360,-0.221919,0.104515,0.0
3,210095290012_R14C03,女,34,NaN,5.4,-0.063122,-0.233871,-0.584804,-0.789165,0.304766,0.644313,-0.343274,10.0
4,209788070040_R12C01,男,49,0.05以下,5.7,-0.088406,-0.143026,-0.066229,0.193179,1.099255,-1.419803,-0.314817,33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2198,209788070041_R16C01,男,52,0.06,5.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2199,209810840123_R09C01,男,48,0.05以下,6.3,0.358983,0.804410,0.554036,1.108380,1.258896,1.673386,0.462905,40.0
2200,209810840125_R05C01,男,54,0.05以下,5.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.2
2201,209798720001_R12C02,男,39,0.14,6.6,-0.173059,-0.167677,-0.477717,-0.127751,-0.176879,0.629214,-0.230596,0.0


In [71]:
df_obs_identified.to_csv("data/observed_data_for_grimagev2.csv", index=False)